# PMS: pulizia di `hotel_reservations_pms.xlsx` (Fase A)

Load, Assess e Clean del singolo file. Per ogni step: il problema, la decisione con il suo motivo, le righe prima e dopo.

- Le celle **VERIFICA** mostrano i numeri su cui si basa la decisione che segue. Non modificano i dati.
- **Regola della Fase A:** un valore diventa NaN solo se è impossibile già da solo. Il recupero dall'altro file (CRM) si fa nella Fase B, dopo il merge.
- Il notebook sta in `Project/cleaning` e legge i dati da `Project/data`.

In [1]:
import pandas as pd

## 1. Load
Si legge il foglio `bookings`: l'altro foglio, `info`, è solo il dizionario delle colonne. Il dato grezzo resta intatto in `raw`, e tutte le modifiche si fanno sulla copia `pms`.

In [2]:
raw = pd.read_excel("../data/raw/hotel_reservations_pms.xlsx", sheet_name="bookings")
pms = raw.copy()
raw.shape

(24480, 20)

## 2. Assess sul dato grezzo (nessuna modifica)

### Mancanti
In `meal` ci sono anche gli `Undefined`. Non compaiono nel dizionario `info` (BB, HB, FB, SC), ma l'articolo originale del dataset (Antonio, de Almeida, Nunes, *Data in Brief* 22, 2019) li definisce "Undefined/SC – no meal package", cioè lo stesso di SC.

In [3]:
print("meal = 'Undefined':", (raw["meal"] == "Undefined").sum())
mancanti = raw.isna().sum()
mancanti[mancanti > 0]

meal = 'Undefined': 238


children    815
meal        490
dtype: int64

### Incoerenze di testo
Le stesse categorie sono scritte in modi diversi.

In [4]:
print("hotel, grafie prima/dopo strip+title:", raw["hotel"].nunique(),
      raw["hotel"].str.strip().str.title().nunique())
print("deposit_type, grafie prima/dopo strip+title:", raw["deposit_type"].nunique(),
      raw["deposit_type"].str.strip().str.title().nunique())
print("arrival_date_month, grafie diverse:", raw["arrival_date_month"].astype(str).nunique())
raw["is_canceled"].astype(str).value_counts()

hotel, grafie prima/dopo strip+title: 10 2
deposit_type, grafie prima/dopo strip+title: 5 3
arrival_date_month, grafie diverse: 47


is_canceled
0      14579
1       8679
no       786
yes      436
Name: count, dtype: int64

### adr salvato come testo, e duplicati
Si contano i valori di `adr` che non si leggono come numero, e quanti di questi non hanno nessuna cifra (segnaposto come "n.d." o "?"). Poi si contano le righe identiche e i codici di prenotazione ripetuti.

In [5]:
adr_testo = raw["adr"].astype(str)
adr_non_numero = pd.to_numeric(raw["adr"], errors="coerce").isna()
print("adr non numerico:", adr_non_numero.sum(),
      "| di cui senza cifre:", (adr_non_numero & ~adr_testo.str.contains("[0-9]")).sum())
print("copie esatte:", raw.duplicated().sum())
print("booking_id ripetuti:", raw["booking_id"].duplicated().sum())

adr non numerico: 608 | di cui senza cifre: 122
copie esatte: 150
booking_id ripetuti: 230


## 3. Clean

### 3.1 Incoerenze: tipi delle colonne
Le tre colonne con il tipo sbagliato contengono testo sporco, quindi non si possono convertire di colpo: si perderebbero valori. Prima si uniforma come sono scritte, poi si converte.
- `arrival_date_month` diventa un numero da 1 a 12.
- `is_canceled` diventa 0/1.
- `adr`: si tolgono simbolo e scritta, la virgola diventa punto, poi si converte in numero. I valori senza cifre diventano mancanti: non si inventa nessun prezzo.

In [6]:
mesi = {"january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
        "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
        "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8,
        "sep": 9, "oct": 10, "nov": 11, "dec": 12}
for n in range(1, 13):
    mesi[str(n)] = n
pms["arrival_date_month"] = pms["arrival_date_month"].astype(str).str.strip().str.lower().map(mesi)

pms["is_canceled"] = pms["is_canceled"].astype(str).str.strip().str.lower().map(
    {"0": 0, "1": 1, "no": 0, "yes": 1})

adr = pms["adr"].astype(str).str.replace("€", "", regex=False).str.replace("EUR", "", regex=False)
adr = adr.str.replace(",", ".", regex=False).str.strip()
pms["adr"] = pd.to_numeric(adr, errors="coerce")

print("mesi non riconosciuti:", pms["arrival_date_month"].isna().sum())
print("is_canceled non riconosciuti:", pms["is_canceled"].isna().sum())
print("adr mancanti dopo la conversione:", pms["adr"].isna().sum())

mesi non riconosciuti: 0
is_canceled non riconosciuti: 0
adr mancanti dopo la conversione: 122


### 3.2 Incoerenze: testo di hotel, deposit_type e meal
- `hotel` e `deposit_type`: si tolgono gli spazi e si porta tutto allo stesso formato (es. "city hotel" → "City Hotel").
- `meal`: "Undefined" e "SC" sono due modi di scrivere "nessun pasto" (fonte: l'articolo originale, vedi Assess). Si ricodifica in SC. Prima si crea la colonna `meal_ricodificato`, che ricorda quali righe sono state cambiate.

In [7]:
pms["hotel"] = pms["hotel"].str.strip().str.title()
pms["deposit_type"] = pms["deposit_type"].str.strip().str.title()

pms["meal_ricodificato"] = pms["meal"] == "Undefined"
pms["meal"] = pms["meal"].replace("Undefined", "SC")
print("meal ricodificati in SC:", pms["meal_ricodificato"].sum())

meal ricodificati in SC: 238


### CRM, letto solo per la verifica
Il CRM non si pulisce e non si unisce: quello è il lavoro della Fase B. Qui serve come seconda fonte per capire la causa dei problemi del PMS. Si tolgono solo le righe senza codice e i codici ripetuti, e `total_price` (con la virgola decimale) si converte in numero.

In [8]:
crm = pd.read_csv("../data/raw/hotel_bookings_crm.csv", sep=";")
crm = crm.dropna(subset=["BookingRef"]).drop_duplicates(subset="BookingRef", keep="last")
crm = crm.set_index("BookingRef")
prezzo_crm = pd.to_numeric(crm["total_price"].astype(str).str.replace(",", ".", regex=False),
                           errors="coerce")
adulti_crm = crm["n_adults"]
crm.shape

(24500, 17)

### 3.3 Duplicati
Prima le copie esatte: basta tenerne una.

In [9]:
righe_prima = len(pms)
pms = pms.drop_duplicates()
print("copie esatte tolte, righe:", righe_prima, "->", len(pms))

copie esatte tolte, righe: 24480 -> 24330


**VERIFICA delle versioni.** Restano prenotazioni con lo stesso codice ma righe diverse. Si confrontano la versione vecchia e quella nuova (in ordine di `record_updated_at`) colonna per colonna. La tabella mostra solo le colonne che cambiano.

In [10]:
pms = pms.sort_values("record_updated_at")
ripetute = pms[pms.duplicated(subset="booking_id", keep=False)]
v_vecchie = ripetute.drop_duplicates(subset="booking_id", keep="first")
v_vecchie = v_vecchie.sort_values("booking_id").reset_index(drop=True)
v_nuove = ripetute.drop_duplicates(subset="booking_id", keep="last")
v_nuove = v_nuove.sort_values("booking_id").reset_index(drop=True)
print("prenotazioni con due versioni:", len(v_nuove))

differenze = {}
for col in pms.columns:
    entrambe_vuote = v_vecchie[col].isna() & v_nuove[col].isna()
    diverse = ((v_vecchie[col] != v_nuove[col]) & ~entrambe_vuote).sum()
    solo_vecchia = (v_nuove[col].isna() & v_vecchie[col].notna()).sum()
    if diverse > 0:
        differenze[col] = {"coppie diverse": diverse, "valore solo nella vecchia": solo_vecchia}
pd.DataFrame(differenze).T

prenotazioni con due versioni: 80


,coppie diverse,valore solo nella vecchia
children,1,1
meal,2,2
booking_changes,12,0
adr,80,0
record_updated_at,80,0


In [11]:
rapporto_adr = v_nuove["adr"] / v_vecchie["adr"]
giorni = (v_nuove["record_updated_at"] - v_vecchie["record_updated_at"]).dt.days
print("adr nuovo / vecchio: min", round(rapporto_adr.min(), 3), "max", round(rapporto_adr.max(), 3))
print("giorni tra le versioni: min", giorni.min(), "max", giorni.max())
print("booking_changes nuovo - vecchio:",
      (v_nuove["booking_changes"] - v_vecchie["booking_changes"]).value_counts().to_dict())
notti_v = v_nuove["stays_in_weekend_nights"] + v_nuove["stays_in_week_nights"]
adr_crm_v = v_nuove["booking_id"].map(prezzo_crm) / notti_v
print("versioni verificabili sul CRM:", adr_crm_v.notna().sum(),
      "| CRM = adr nuovo:", ((adr_crm_v / v_nuove["adr"] - 1).abs() < 0.01).sum(),
      "| CRM = adr vecchio:", ((adr_crm_v / v_vecchie["adr"] - 1).abs() < 0.01).sum())

adr nuovo / vecchio: min 1.118 max 1.33
giorni tra le versioni: min 2 max 39
booking_changes nuovo - vecchio: {0: 68, 1: 12}
versioni verificabili sul CRM: 77 | CRM = adr nuovo: 76 | CRM = adr vecchio: 0


**Decisione:** vince la versione più recente, e il CRM lo conferma. In 3 casi però la nuova ha un buco che la vecchia non ha (1 in `children`, 2 in `meal`). Con `combine_first` si tiene la nuova e si riempiono solo i suoi buchi con la vecchia: è la regola "most complete" del golden record.

È un'ipotesi. In quei 3 casi `booking_changes` vale 0 e, tra i valori noti, `meal` e `children` non cambiano mai da una versione all'altra.

In [12]:
colonne = pms.columns
righe_prima = len(pms)
ultime = pms.drop_duplicates(subset="booking_id", keep="last").set_index("booking_id")
vecchie = pms[pms.duplicated(subset="booking_id", keep="last")].set_index("booking_id")
pms = ultime.combine_first(vecchie).reset_index()
pms = pms[colonne]
print("versioni unite, righe:", righe_prima, "->", len(pms))

versioni unite, righe: 24330 -> 24250


### 3.4 Valori impossibili
**VERIFICA:** quanti sono e cosa dice il CRM sulla loro causa. `adr_crm = total_price / notti` è la formula che userà il merge, quindi prima si controlla che funzioni sulle righe normali.

In [13]:
notti = pms["stays_in_weekend_nights"] + pms["stays_in_week_nights"]
prezzo = pms["booking_id"].map(prezzo_crm)
adr_crm = prezzo / notti
verificabile = prezzo.notna() & (notti > 0)
normali = verificabile & (pms["adr"] > 0) & (pms["adr"] <= 1000)
print("formula total_price / notti = adr sulle righe normali:",
      round(((adr_crm / pms["adr"] - 1).abs() < 0.01)[normali].mean(), 3))

negativi = pms["adr"] < 0
print("adr negativo:", negativi.sum(), "| verificabili:", (negativi & verificabile).sum(),
      "| CRM = stesso valore in positivo:",
      ((adr_crm / -pms["adr"] - 1).abs() < 0.01)[negativi & verificabile].sum())

alti = pms["adr"] > 1000
print("adr sopra 1000:", alti.sum(), "| massimo sotto 1000:", pms["adr"][pms["adr"] <= 1000].max(),
      "| minimo sopra 1000:", pms["adr"][alti].min())
print("   verificabili:", (alti & verificabile).sum(),
      "| CRM = adr / 100:", ((pms["adr"] / 100 / adr_crm - 1).abs() < 0.01)[alti & verificabile].sum(),
      "| CRM = adr:", ((adr_crm / pms["adr"] - 1).abs() < 0.01)[alti & verificabile].sum())

zero = pms["adr"] == 0
print("adr = 0:", zero.sum(), "| con prezzo CRM:", (zero & prezzo.notna()).sum(),
      "| CRM = 0:", (zero & (prezzo == 0)).sum())

formula total_price / notti = adr sulle righe normali: 0.988
adr negativo: 40 | verificabili: 39 | CRM = stesso valore in positivo: 39
adr sopra 1000: 31 | massimo sotto 1000: 387.0 | minimo sopra 1000: 3511.0
   verificabili: 27 | CRM = adr / 100: 26 | CRM = adr: 1
adr = 0: 377 | con prezzo CRM: 362 | CRM = 0: 362


In [14]:
meno_uno = pms["stays_in_week_nights"] < 0
week_ricostruite = (prezzo / pms["adr"]).round() - pms["stays_in_weekend_nights"]
print("stays_in_week_nights negativo:", meno_uno.sum(),
      "| valori:", pms["stays_in_week_nights"][meno_uno].value_counts().to_dict())
print("   valore vero ricostruito dal CRM:",
      week_ricostruite[meno_uno & (pms["adr"] > 0)].value_counts().sort_index().to_dict())

print("prenotazioni a 0 notti:", (notti == 0).sum(),
      "| di cui solo per weekend + (-1):", ((notti == 0) & meno_uno).sum())

ospiti = pms["adults"] + pms["children"] + pms["babies"]
print("senza ospiti:", (ospiti == 0).sum(),
      "| 0 adulti, 0 neonati e children mancante:",
      ((pms["adults"] == 0) & (pms["babies"] == 0) & pms["children"].isna()).sum())

molti = pms["adults"] > 10
print("più di 10 adulti:", molti.sum())
pms["booking_id"].map(adulti_crm)[molti].value_counts(dropna=False)

stays_in_week_nights negativo: 35 | valori: {-1: 35}
   valore vero ricostruito dal CRM: {1.0: 10, 2.0: 10, 3.0: 4, 4.0: 3, 5.0: 2, 6.0: 2}
prenotazioni a 0 notti: 159 | di cui solo per weekend + (-1): 12
senza ospiti: 31 | 0 adulti, 0 neonati e children mancante: 7
più di 10 adulti: 23


booking_id
2.0     15
1.0      4
NaN      2
26.0     1
55.0     1
Name: count, dtype: int64

**VERIFICA della coerenza** tra colonne che dipendono l'una dall'altra.

In [15]:
data = pd.to_datetime(pms["arrival_date_year"].astype(str) + "-"
                      + pms["arrival_date_month"].astype(str) + "-"
                      + pms["arrival_date_day_of_month"].astype(str), errors="coerce")
print("date di arrivo non valide:", data.isna().sum(), "| dal", data.min().date(), "al", data.max().date())
contatori = ["adults", "children", "babies", "booking_changes", "required_car_parking_spaces",
             "total_of_special_requests", "stays_in_weekend_nights"]
print("valori negativi nei contatori:", (pms[contatori] < 0).sum().sum())
print("aggiornati oltre 30 giorni dopo la partenza:",
      ((pms["record_updated_at"] - data).dt.days > notti + 30).sum())
print("parcheggi più degli ospiti:", (pms["required_car_parking_spaces"] > ospiti).sum(),
      "| cancellate con parcheggi:",
      ((pms["is_canceled"] == 1) & (pms["required_car_parking_spaces"] > 0)).sum())

date di arrivo non valide: 0 | dal 2015-07-01 al 2017-08-31
valori negativi nei contatori: 0
aggiornati oltre 30 giorni dopo la partenza: 0
parcheggi più degli ospiti: 2 | cancellate con parcheggi: 0


**Notti del weekend.** Si conta quante notti di una certa coppia di giorni cadono nel soggiorno, sapendo il giorno di arrivo (0 = lunedì … 6 = domenica) e le notti totali. Ogni settimana intera ne contiene 2. Nei giorni rimasti, un giorno g cade nel soggiorno se dista dall'arrivo meno dei giorni rimasti.

In [16]:
giorno = data.dt.dayofweek
settimane = notti // 7
resto = notti % 7
sab_dom = settimane * 2 + ((5 - giorno) % 7 < resto).astype(int) + ((6 - giorno) % 7 < resto).astype(int)
dom_lun = settimane * 2 + ((6 - giorno) % 7 < resto).astype(int) + ((7 - giorno) % 7 < resto).astype(int)
controllabili = ~meno_uno
print("notti weekend = notti di sabato e domenica:",
      round((sab_dom == pms["stays_in_weekend_nights"])[controllabili].mean(), 3))
print("notti weekend = notti di domenica e lunedì:",
      round((dom_lun == pms["stays_in_weekend_nights"])[controllabili].mean(), 3))

notti weekend = notti di sabato e domenica: 0.498
notti weekend = notti di domenica e lunedì: 1.0


Risultato: la regola è coerente, ma spostata di un giorno (domenica e lunedì). Non si corregge nulla. Nella Fase B servirà per controllare le notti ricostruite.

**Decisioni.** Prima di cambiare un valore si crea la colonna che lo segna. Così, nella Fase B e nei grafici prima/dopo, si distinguono i mancanti originali da quelli creati qui.

**adr negativo o sopra 1000 euro.** Un prezzo per notte non può essere negativo. Sopra 387 euro c'è un salto fino a 3.511, quindi la soglia cade nel vuoto e non è arbitraria. Questi valori diventano NaN, e nella Fase B si ricalcolano da `total_price / notti` del CRM. Le righe senza CRM resteranno NaN: è una perdita voluta, perché la causa non è provata riga per riga. BK323523 (5.400 euro per 1 notte) è confermato dal CRM: nella Fase B tornerà a 5.400 e andrà segnato come outlier.

**adr = 0.** Si tiene, perché anche il CRM dice 0 (soggiorni gratuiti, pacchetti di gruppi o agenzie, uso diurno). Si segna con `adr_zero`: nell'analisi dei prezzi queste righe vanno escluse, altrimenti abbassano la media.

In [17]:
pms["adr_anomalo"] = (pms["adr"] < 0) | (pms["adr"] > 1000)
pms["adr"] = pms["adr"].where(pms["adr"].between(0, 1000))
print("adr messi a NaN:", pms["adr_anomalo"].sum())

pms["adr_zero"] = pms["adr"] == 0
print("adr = 0 segnati:", pms["adr_zero"].sum())

adr messi a NaN: 71
adr = 0 segnati: 377


**stays_in_week_nights = -1.** È un codice per "non noto", non un segno sbagliato: il valore vero va da 1 a 6. Diventa NaN. Nella Fase B si ricostruisce prima dalle date del CRM (solo per i check-out), poi da `total_price / adr` meno le notti del weekend.

**0 notti.** Si tengono: sono soggiorni di un giorno, quasi tutti in check-out nel CRM. Il flag si calcola dopo aver tolto il -1, così le righe con weekend 1 e week -1 non vengono contate.

In [18]:
pms["week_nights_mancante"] = pms["stays_in_week_nights"] < 0
pms["stays_in_week_nights"] = pms["stays_in_week_nights"].where(pms["stays_in_week_nights"] >= 0)
print("notti feriali messe a NaN:", pms["week_nights_mancante"].sum())

pms["zero_notti"] = (pms["stays_in_weekend_nights"] + pms["stays_in_week_nights"]) == 0
print("0 notti segnate:", pms["zero_notti"].sum())

notti feriali messe a NaN: 35
0 notti segnate: 147


**Senza ospiti.** Si tengono e si segnano. Il numero vero di ospiti non si conosce, e il CRM non aiuta: dice anche lui 0, ma probabilmente copia lo stesso dato d'origine, quindi non è una conferma indipendente. Il flag si calcola prima di riempire `children`: le righe con `children` mancante non vengono segnate, perché il loro "zero bambini" sarebbe solo una stima.

**Più di 10 adulti.** Nessuna modifica in questa fase. Il valore non è impossibile da solo: esistono gruppi veri, e alcuni sono confermati dal CRM. Che l'adr sia maggiore di 0 è solo un indizio. Decide il CRM nella Fase B, riga per riga.

In [19]:
pms["senza_ospiti"] = (pms["adults"] + pms["children"] + pms["babies"]) == 0
print("senza ospiti segnati:", pms["senza_ospiti"].sum())

senza ospiti segnati: 31


### 3.5 Mancanti
Si trattano per ultimi, perché lo step sui valori impossibili ne crea di nuovi.

In [20]:
mancanti = pms.isna().sum()
mancanti[mancanti > 0]

stays_in_week_nights     35
children                809
meal                    483
adr                     191
dtype: int64

**VERIFICA:** dove sono i mancanti di `children` e `meal`, e com'è fatta la distribuzione di `children`.

In [21]:
children_vuoto = pms["children"].isna()
quota_mese = children_vuoto[pms["hotel"] == "City Hotel"].groupby(pms["arrival_date_month"]).mean()
quota_anno = children_vuoto[pms["hotel"] == "City Hotel"].groupby(pms["arrival_date_year"]).mean()
print("children mancante nel City Hotel, per mese: min", round(quota_mese.min(), 3),
      "max", round(quota_mese.max(), 3),
      "| per anno: min", round(quota_anno.min(), 3), "max", round(quota_anno.max(), 3))
print("children noti: media", round(pms["children"].mean(), 2), "| mediana", pms["children"].median(),
      "| moda", pms["children"].mode()[0], "| quota di zeri",
      round((pms["children"] == 0).sum() / pms["children"].notna().sum(), 3))
pd.DataFrame({"children mancante": children_vuoto.groupby(pms["hotel"]).mean(),
              "meal mancante": pms["meal"].isna().groupby(pms["hotel"]).mean()}).round(3)

children mancante nel City Hotel, per mese: min 0.035 max 0.062 | per anno: min 0.047 max 0.052
children noti: media 0.1 | mediana 0.0 | moda 0.0 | quota di zeri 0.93


,children mancante,meal mancante
hotel,,
City Hotel,0.05,0.018
Resort Hotel,0.00,0.023


**Decisioni.**

**meal.** I mancanti veri restano vuoti. Non si sa quale pasto sia stato prenotato, e la moda (BB) attribuirebbe un pasto sbagliato a una parte delle righe.

**children.** I mancanti sono solo nel City Hotel e sono distribuiti in modo uniforme tra mesi e anni. Non si possono dedurre da altre colonne, e togliere le righe toglierebbe solo prenotazioni del City Hotel. Si riempiono con 0: la media (0,1) darebbe un numero di bambini che non esiste, mentre moda e mediana coincidono e valgono entrambe 0. È una **STIMA**: prima di riempire si crea la colonna `children_missing`, che ricorda quali righe sono state riempite.

**adr.** I mancanti, sia gli originali sia quelli creati in 3.4, si deducono dopo il merge.

In [22]:
pms["children_missing"] = pms["children"].isna()
pms["children"] = pms["children"].fillna(0)
print("children riempiti con 0:", pms["children_missing"].sum())

children riempiti con 0: 809


## Fine della Fase A
Ordine finale per codice di prenotazione, così l'output è stabile. Sotto: la forma finale e quante righe segna ogni flag.

In [23]:
pms = pms.sort_values("booking_id").reset_index(drop=True)
print("righe, colonne:", pms.shape)
flag = ["meal_ricodificato", "adr_anomalo", "adr_zero", "week_nights_mancante",
        "zero_notti", "senza_ospiti", "children_missing"]
pms[flag].sum()

righe, colonne: (24250, 27)


meal_ricodificato       236
adr_anomalo              71
adr_zero                377
week_nights_mancante     35
zero_notti              147
senza_ospiti             31
children_missing        809
dtype: int64

## Export
La tabella pulita si salva in `data/processed/`, mentre i file grezzi in `data/raw/` restano intatti. La Fase B la rilegge con `pd.read_csv`.

In [ ]:
pms.to_csv("../data/processed/pms_clean.csv", index=False)
print("salvato ../data/processed/pms_clean.csv:", pms.shape)

## Note per la Fase B (merge con il CRM)

Prima di usare il CRM per riempire, si controlla che le colonne in comune tornino:
- `hotel` / `property`;
- data di arrivo / `checkin_date`;
- `adults` / `n_adults`;
- `is_canceled` / `reservation_status`;
- notti / differenza tra le date del CRM (solo per i check-out);
- `total_price` / adr per notti;
- la regola domenica-lunedì sulle notti ricostruite;
- arrivo meno `lead_time` non successivo a `record_updated_at`;
- `is_repeated_guest` / prenotazioni precedenti.

Da fare:
- **adr:**
  - ricalcolo da `total_price / notti`;
  - le righe con adr mancante e 0 notti diventano 0, perché tutte le righe a 0 notti con adr noto hanno adr 0;
  - BK323523 va segnato come outlier;
  - in circa 278 righe il CRM è più alto del PMS (dal 15% al 58%): lì la deduzione va controllata.
- **week_nights:** prima le date del CRM per i check-out, poi `total_price / adr` meno il weekend, solo se il risultato è intero. I valori ricostruiti sono una STIMA.
- **Adulti oltre 10:** confronto con `n_adults` del CRM.
- **Conflitti già visti:**
  - circa 56 righe con adulti diversi tra i due file;
  - circa 46 date di arrivo spostate di qualche giorno;
  - date del CRM in formati misti.